# 01 – EDA and Modeling Decisions

End-to-end exploratory data analysis (EDA) and key modeling decisions for the *Credit Card Fraud Detection* dataset (Kaggle: `mlg-ulb/creditcardfraud`).

This notebook is designed to run **end-to-end** once `data/raw/creditcard.csv` is available.


## A) Business Framing and Anomaly Detection Perspective

- **Business goal**: Detect fraudulent credit card transactions early so they can be blocked or investigated.
- **Operational constraint**: Fraud is rare but costly; we must balance catching as many frauds as possible with avoiding excessive false alarms.
- **Modeling view**: Treat fraud detection as **anomaly detection** on transaction features:
  - Train unsupervised / one-class models on normal transactions.
  - Use the learned notion of normal to assign anomaly scores to new transactions.

Key implications:
- Majority of transactions are legitimate; models must be robust to extreme class imbalance.
- False negatives (missed fraud) are typically more costly than false positives, but both matter.
- We emphasize **ranking quality** (PR-AUC, ROC-AUC) and **thresholded performance** (precision/recall/F1).


In [ ]:
# Setup: imports, paths, and reproducibility
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from shared.utils import set_global_seed
from ml import preprocess as ml_preprocess

DATA_PATH = Path('data/raw/creditcard.csv')

set_global_seed(42)
plt.style.use('default')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Expected dataset at {DATA_PATH}. Please download it first.'
    )

df = ml_preprocess.load_raw_data(DATA_PATH)
df.head()


## B) Dataset Facts and Class Imbalance

The Kaggle dataset contains European credit card transactions from September 2013.

We summarize basic facts and quantify the imbalance in the `Class` label (1 = fraud, 0 = non-fraud).


In [ ]:
# Basic shape and class distribution
n_rows, n_cols = df.shape
class_counts = df['Class'].value_counts().sort_index()
class_frac = df['Class'].value_counts(normalize=True).sort_index() * 100

print(f'Rows: {n_rows}, Columns: {n_cols}')
print('Class distribution (counts):')
print(class_counts)
print('\nClass distribution (%):')
print(class_frac.round(4))

fraud_rate = class_frac.loc[1]
print(f'\nEstimated fraud rate: {fraud_rate:.4f}% of transactions are fraud.')


## C) Feature Overview and Data Quality Checks

Columns:

- `Time`: Seconds elapsed between this transaction and the first transaction in the dataset.
- `Amount`: Transaction amount (Euro).
- `Class`: Target label (1 = fraud, 0 = non-fraud).
- `V1`–`V28`: Numeric features derived via PCA for confidentiality.

We verify data types and check for missing values.


In [ ]:
# Data types and missing values
print('Column dtypes:\n')
print(df.dtypes)

print('\nMissing values per column:\n')
missing = df.isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else 'No missing values detected.')


## D) Leakage-Safe Time-Based Split

We want to mimic a real-world scenario where models are trained on past data and evaluated on future data.

Strategy:

- Sort by `Time` in ascending order.
- Split into:
  - Train: 60% of earliest transactions.
  - Validation: next 20%.
  - Test: final 20%.
- Ensure no future information leaks into the past via random shuffling.

We reuse the time-based split logic from `ml.preprocess.time_based_split` to stay consistent with the training pipeline.


In [ ]:
# Time-based split using the shared preprocessing utility
train_df, val_df, test_df = ml_preprocess.time_based_split(
    df,
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2,
    sort_by='Time',
)

print('Split sizes:')
print({'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})

def summarize_time(split_df, name):
    print(f'{name} Time range: {split_df["Time"].min()} -> {split_df["Time"].max()}')

summarize_time(train_df, 'Train')
summarize_time(val_df, 'Val')
summarize_time(test_df, 'Test')


### Time Distribution Across Splits

We visualize how transactions (especially frauds) are distributed over time across train/validation/test splits.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, split_df, title in zip(
    axes,
    [train_df, val_df, test_df],
    ['Train', 'Validation', 'Test'],
):
    fraud_times = split_df.loc[split_df['Class'] == 1, 'Time']
    ax.hist(split_df['Time'], bins=50, alpha=0.5, label='All', color='tab:blue')
    if not fraud_times.empty:
        ax.hist(
            fraud_times,
            bins=50,
            alpha=0.7,
            label='Fraud',
            color='tab:red',
        )
    ax.set_title(f'{title} Time Distribution')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()


## E) Distribution Plots

We inspect key distributions to understand differences between fraud and non-fraud transactions.

Focus:
- `Amount` distribution (with log scale) by class.
- `Time` distribution of frauds.
- PCA components (V1–V28) summary statistics.


In [ ]:
# Amount distribution by class (log-scaled x-axis)
fig, ax = plt.subplots(figsize=(8, 4))
non_fraud = df[df['Class'] == 0]['Amount']
fraud = df[df['Class'] == 1]['Amount']

ax.hist(non_fraud, bins=50, alpha=0.5, label='Non-fraud', color='tab:blue')
ax.hist(fraud, bins=50, alpha=0.7, label='Fraud', color='tab:red')
ax.set_xscale('log')
ax.set_xlabel('Amount (log scale)')
ax.set_ylabel('Count')
ax.set_title('Amount Distribution by Class')
ax.legend()
plt.tight_layout()
plt.show()

# Time distribution of frauds only
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.loc[df['Class'] == 1, 'Time'], bins=50, color='tab:red', alpha=0.8)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Fraud count')
ax.set_title('Time Distribution of Fraud Transactions')
plt.tight_layout()
plt.show()

# PCA components summary statistics
pca_cols = [f'V{i}' for i in range(1, 29)]
pca_summary = df[pca_cols].describe().T
pca_summary.head()


## F) Correlation Heatmap (PCA Features)

We examine correlations among PCA components. Since they are derived features,
correlation patterns may not map directly to interpretable business concepts,
but they still inform us about structure in the transformed feature space.


In [ ]:
corr = df[['Amount', 'Time'] + [f'V{i}' for i in range(1, 29)]].corr()

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap (Amount, Time, PCA Components)')
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=6)
fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## G) Supervised Proxy for Feature Importance

Although our primary models are anomaly detectors trained only on normal data,
we can train a simple supervised proxy model using the `Class` labels to gain
insight into which PCA components appear most influential.

Caveats:

- PCA components (`V1`–`V28`) are linear combinations of original, hidden features.
- Importance on these components does not translate directly to actionable business features.
- Still, relative importance can inform which components to monitor.

We use:

- LogisticRegression with `class_weight='balanced'` as a supervised baseline.
- The same preprocessing as the anomaly models (RobustScaler on Time/Amount).
- `sklearn.inspection.permutation_importance` on the validation split.


In [ ]:
# Build preprocessor and supervised proxy pipeline
feature_cols = [c for c in train_df.columns if c != 'Class']
preprocessor = ml_preprocess.build_preprocessor()

log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    n_jobs=-1,
)
proxy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', log_reg),
])

proxy_pipeline.fit(train_df[feature_cols], train_df['Class'])

perm_result = permutation_importance(
    proxy_pipeline,
    val_df[feature_cols],
    val_df['Class'],
    n_repeats=5,
    random_state=42,
    scoring='roc_auc',
)

importances = pd.DataFrame(
    {
        'feature': feature_cols,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std,
    }
).sort_values('importance_mean', ascending=False)

importances.head(15)


The table above shows which components (Vx), along with `Time` and `Amount`,
contribute most to the supervised proxy's discrimination power.

In downstream monitoring, we will pay particular attention to:

- The most influential PCA components (by permutation importance).
- The distribution of `Amount` and `Time`.
- How these features shift over time and relate to predicted fraud scores.


## H) Metric Choice and Thresholding Strategy

Given the extreme class imbalance, we focus on metrics that emphasize the positive (fraud) class:

- ROC-AUC: ranking quality overall, but can remain high even if precision is low.
- PR-AUC (Precision–Recall AUC): more informative under heavy imbalance.
- Precision: of the transactions flagged as fraud, how many are truly fraud.
- Recall: of the truly fraudulent transactions, how many we catch.
- F1: harmonic mean of precision and recall at a given threshold.

Thresholding strategy for anomaly scores (see `context/METRICS.md` and `ml.evaluate.choose_threshold`):

- Use validation data to sweep possible thresholds on anomaly scores.
- Prefer thresholds that simultaneously achieve:
  - High precision (e.g., ≥ 0.90) to limit false alarms.
  - Reasonable recall (e.g., ≥ 0.75) to catch most frauds.
- If no threshold satisfies both constraints, choose the one with the best F1 and document the trade-off.


## I) Reproducibility

We ensure that experiments are reproducible by:

- Fixing random seeds across NumPy, Python's `random`, and model libraries via `shared.utils.set_global_seed`.
- Recording package versions and Python version.
- Using deterministic time-based splits (no shuffling).


In [ ]:
import sys
import sklearn
import pyod

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('pyod:', pyod.__version__)


## J) What We Will Monitor in Production

To operate these models safely and effectively in production, we plan to monitor:

1. Prediction volume:
   - Number of scored transactions per unit time.
   - Breakdown by predicted class and score distribution.

2. Observed fraud rate:
   - Fraction of flagged transactions later confirmed as fraudulent (where labels are available).
   - Drift in observed fraud rate over time.

3. Latency:
   - End-to-end scoring latency per transaction.
   - Tail latencies (p95, p99) to ensure real-time constraints are met.

4. Data drift (planned):
   - Statistical tests (PSI, KS) for key features: `Amount`, `Time`, top PCA components.
   - Distribution of anomaly scores compared to training/validation baselines.

5. Model performance (where labels are available):
   - PR-AUC, ROC-AUC, precision, recall, and F1 on recent labeled windows.
   - Stability of calibrated thresholds and whether precision/recall constraints remain satisfied.

These monitoring signals will inform when to retrain models, adjust thresholds, or investigate upstream issues.
